# Aula 04 - Notebook: Implementação de Conectivos Lógicos e Permissivos de Partida

Neste notebook implementamos as funções de avaliação lógica proposicional completas (AND, OR, NOT, XOR) e construímos os blocos de permissivos de partida (*Start Permissives*) e intertravamento contínuo para o Forno de Torra da planta de processamento de amendoim.

In [ ]:
from typing import Dict
import pandas as pd
import itertools

# Operadores Fundamentais da Lógica Proposicional
def NOT(p: bool) -> bool:
    return not p

def AND(p: bool, q: bool) -> bool:
    return p and q

def OR(p: bool, q: bool) -> bool:
    return p or q

def XOR(p: bool, q: bool) -> bool:
    return p ^ q

print("Operadores lógicos proposicionais carregados com sucesso.")

## Bloco Lógico de Permissivo do Forno de Torra

In [ ]:
def permissivo_forno(f2: bool, t2: bool, e1: bool, auto_mode: bool, manual_mode: bool) -> Dict[str, bool]:
    # Condição de modo exclusivo (Auto XOR Manual)
    modo_valido = XOR(auto_mode, manual_mode)
    
    # Condição combinada de permissivo
    permissivo = (f2 and 
                  NOT(t2) and 
                  NOT(e1) and 
                  modo_valido)
    
    # Condição de trip imediato
    trip = NOT(f2) or t2 or e1
    
    return {
        'Permissivo_Habilitado': permissivo,
        'Trip_Ativo': trip,
        'Modo_Valido': modo_valido
    }

# Teste com diferentes cenários de campo
cenarios = [
    {"cenario": "Operação Normal (Auto)", "args": (True, False, False, True, False)},
    {"cenario": "Falha de Exaustão", "args": (False, False, False, True, False)},
    {"cenario": "Sobreaquecimento", "args": (True, True, False, True, False)},
    {"cenario": "Parada de Emergência Ativa", "args": (True, False, True, True, False)},
    {"cenario": "Conflito de Modo (Auto e Manual juntos)", "args": (True, False, False, True, True)},
]

resultados = []
for c in cenarios:
    res = permissivo_forno(*c["args"])
    resultados.append({
        "Cenário": c["cenario"],
        "Permissivo": res["Permissivo_Habilitado"],
        "Trip Ativo": res["Trip_Ativo"],
        "Modo Válido": res["Modo_Valido"]
    })

pd.DataFrame(resultados)

## Geração Automática de Tabela-Verdade para Validação Exhaustiva

In [ ]:
variaveis = ['f2', 't2', 'e1']
tabela = []

for combo in itertools.product([False, True], repeat=len(variaveis)):
    st = dict(zip(variaveis, combo))
    res = permissivo_forno(st['f2'], st['t2'], st['e1'], True, False)
    row = {**st, 'Permissivo': res['Permissivo_Habilitado'], 'Trip': res['Trip_Ativo']}
    tabela.append(row)

df_tv = pd.DataFrame(tabela)
print(f"Total de combinações avaliadas: {len(df_tv)}")
print(f"Combinações seguras que liberam o forno: {df_tv['Permissivo'].sum()}")
print(df_tv.head(8))
